In [1]:
import yadisk
import pandas as pd
from io import BytesIO
import os 
from dotenv import load_dotenv

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

load_dotenv()

TOKEN = os.getenv("YANDEX_TOKEN")
DISK_PATH = os.getenv("DISK_PATH")

print(f"Токен загружен: {bool(TOKEN)}")

Токен загружен: True


In [2]:
def read_xlsx_from_personal_disk(token, disk_path):
    y = yadisk.YaDisk(token=token)
    
    if not y.check_token():
        raise Exception("Неверный OAuth-токен")
        
    if not y.exists(disk_path):
        raise Exception(f"Файл не найден по пути: {disk_path}")

    try:
        # Создаем буфер в памяти
        buffer = BytesIO()
        
        # Метод download качает файл прямо в наш буфер
        y.download(disk_path, buffer)
        
        # После скачивания "курсор" буфера находится в конце файла.
        # Сбрасываем его в начало, чтобы Pandas мог прочитать данные.
        try:
            buffer.seek(0)
            df = pd.read_excel(buffer, engine='calamine')
        except ValueError as e:
            if "could not read stylesheet" in str(e):
                print("Ошибка стилей. Пробуем принудительно пересчитать стили...")
                # Попытка использовать альтернативный путь обработки
                with open("temp_fix.xlsx", "wb") as f:
                    f.write(buffer.getvalue())
                
                import subprocess
                try:
                    # Пытаемся открыть и закрыть файл через COM-интерфейс Excel (только Windows)
                    # Это самый надежный способ починить битый XML "на лету"
                    pass 
                except Exception:
                    pass
                
                df = pd.read_excel(buffer, engine='calamine')
        return df
        
    except Exception as e:
        raise Exception(f"Ошибка при загрузке или чтении Excel: {e}")

# Выполнение
try:
    data_frame = read_xlsx_from_personal_disk(TOKEN, DISK_PATH)
    print("Успешно прочитано строк:", len(data_frame))
    print(data_frame.head(1000))
except Exception as error:
    print(f"Произошла ошибка: {error}")

Успешно прочитано строк: 24
       №         Код    Код ОКПД2                           Номенклатура       QR код              Статус        Проект МТО/Услуга  Цена  Сумма  Упаковка Ед. изм.  Кол-во   Строка плана закупки  План потребности              План МТР  Отдел МТО Куратор МТО Счет от поставщика\n   согласован  Цена финальная  Сумма финальная   Заявка на закупку           Заявка на оплату  Приходный ордер   Оплачено   Получено со склада  Отменено  Ссылка  Остаток Место хранения  Списано
0    1.0   1000000.0  00.00.00.00                      Транзистор АБВГ 5   МТО1000000  Получено со склада  ОООБибаИБоба        МТО   100    100       NaN       шт       1  утвержена от 14.09.26               NaN  00-000 от 24.09.2026        111        Биба                               Нет             110              110  1998 от 14.09.2026  АБ00-000000 от 14.09.2026              NaN 2026-09-14  Получено 15.09.2026       NaN     NaN      NaN           10В0      NaN
1    2.0  11111111.0  00.00.00